<a href="https://colab.research.google.com/github/Nithya2405/ai-engineering-workbench/blob/main/06_supervisor_orchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🧠 The Day 12 Concept: Multi-Agent Systems (MAS)

Instead of one giant, confused prompt, I split the brain:

1. **The Researcher:** Specialized only in fetching live web data.
2. **The Analyst:** Specialized in cleaning data and performing calculations.
3. **The Supervisor:** The "Project Manager" who decides who should talk when.

---

### 💻 Day 12: The Multi-Agent Supervisor

This code implements a **Router** that identifies the intent and handsoff the task to the correct specialized function.

In [4]:
!pip install groq
!pip install tavily
import os
from groq import Groq
from tavily import TavilyClient
from google.colab import userdata
import json

# 1. Setup

In [5]:
client = Groq(api_key=userdata.get('GROQ_API_KEY'))
tavily = TavilyClient(api_key=userdata.get('TAVILY_API_KEY'))

# 2. Define specialized "Worker" functions

In [6]:
def researcher_worker(query):
    print(f"🕵️ [RESEARCHER] Finding: {query}")
    return tavily.search(query=query, max_results=2)['results'][0]['content']

def analyst_worker(data):
    print(f"📊 [ANALYST] Processing data...")
    # Simulated analysis logic
    return f"Analysis complete. Key Insight: {data[:100]}..."

# 3. The Supervisor (The Brain)

In [7]:
def supervisor_orchestrator(user_request):
    print(f"👑 [SUPERVISOR] Task: {user_request}\n" + "="*40)


    # Step 1: Decision - Who should handle this?
    decision_prompt = f"""
    You are a Supervisor Agent. Based on the user request, decide if we need 'RESEARCH', 'ANALYSIS', or 'BOTH'.
    Request: {user_request}
    Response format: ONLY the word.
    """

    plan = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "system", "content": decision_prompt}]
    ).choices[0].message.content.strip()

    # Step 2: Execution based on Plan
    context = ""
    if "RESEARCH" in plan or "BOTH" in plan:
        context = researcher_worker(user_request)

    if "ANALYSIS" in plan or "BOTH" in plan:
        report = analyst_worker(context if context else user_request)
        print(f"✅ Final Result: {report}")
    else:
        print(f"✅ Final Result: {context}")

# Test the Orchestration

In [8]:
supervisor_orchestrator("Research the current NVIDIA stock price and give me a summary.")

👑 [SUPERVISOR] Task: Research the current NVIDIA stock price and give me a summary.
🕵️ [RESEARCHER] Finding: Research the current NVIDIA stock price and give me a summary.
✅ Final Result: The NVIDIA stock price today is 167.52 USD. What Stock Exchange Does NVIDIA Trade On? NVIDIA is listed and trades on the Nasdaq Stock Exchange.
